# 模型分布诊断复核
只读，历史 revision 按审计截点保留；所有筛查均不是晋级 OOS。
运行提取器会重新查询本地 SSOT；仅阅读已保存 JSON 不访问生产。

In [1]:
from pathlib import Path
import json, math
p = Path('/Users/xiong/weather/tmp/model_distribution_diagnosis_20260910')
summary = json.loads((p/'analysis.json').read_text())
print(summary['sensitivity'])


[{'arm': 'baseline5', 'predictions': 8593, 'station_days': 899, 'dates': 20, 'stations': 45, 'stations_better_nll': 0, 'nll': 1.7755812081975442, 'crps_c': 0.7302415177647053, 'cover80': 0.7650102739924763, 'cover95': 0.9040215716800811, 'nll_gain': 0.0, 'crps_gain': 0.0, 'date_block4_bootstrap_nll_gain_ci95': [0.0, 0.0]}, {'arm': 'spread15', 'predictions': 8593, 'station_days': 899, 'dates': 20, 'stations': 45, 'stations_better_nll': 34, 'nll': 1.7336022276647103, 'crps_c': 0.7296467618134349, 'cover80': 0.7981367353447442, 'cover95': 0.9240375370519975, 'nll_gain': 0.04197898053283393, 'crps_gain': 0.0005947559512702279, 'date_block4_bootstrap_nll_gain_ci95': [0.0235090825557963, 0.06283262301426992]}, {'arm': 'spread30', 'predictions': 8593, 'station_days': 899, 'dates': 20, 'stations': 45, 'stations_better_nll': 33, 'nll': 1.729087020447434, 'crps_c': 0.7300818943987676, 'cover80': 0.8056869128893601, 'cover95': 0.927743795569157, 'nll_gain': 0.04649418775011012, 'crps_gain': 0.000

In [2]:
r = 2**(-1/5)
neff = (1+r)/(1-r)
print(neff, 1/math.sqrt(2*(neff-1)))


14.450047917745149 0.19280712836733696


In [3]:
bucket = json.loads((p/'exact_bucket_screen.json').read_text())
print(bucket['summary'])


[{'arm': 'baseline5', 'scope_day_predictions': 1866, 'station_days': 180, 'nll': 1.7172821554589566, 'peak_hit': 0.3488324175824176, 'crps_c': 0.7437040704432891, 'nll_gain': 0.0, 'crps_gain_c': 0.0, 'peak_hit_change_pp': 0.0}, {'arm': 'spread15', 'scope_day_predictions': 1866, 'station_days': 180, 'nll': 1.686963974319558, 'peak_hit': 0.3404028995695662, 'crps_c': 0.7438124215103545, 'nll_gain': 0.03031818113939866, 'crps_gain_c': -0.00010835106706541531, 'peak_hit_change_pp': -0.8429518012851411}, {'arm': 'spread30', 'scope_day_predictions': 1866, 'station_days': 180, 'nll': 1.6844351689333128, 'peak_hit': 0.341092595675929, 'crps_c': 0.7443607724393865, 'nll_gain': 0.03284698652564377, 'crps_gain_c': -0.000656701996097353, 'peak_hit_change_pp': -0.7739821906488642}, {'arm': 'rms_denominator5', 'scope_day_predictions': 1866, 'station_days': 180, 'nll': 1.7315716310096314, 'peak_hit': 0.35417158458825126, 'crps_c': 0.7421409305401555, 'nll_gain': -0.014289475550674835, 'crps_gain_c': 

In [4]:
serving = json.loads((p/'serving_analysis.json').read_text())
print(serving['matched_serving_pairs'])


[{'cohort': 'all', 'pairs': 155, 'stations': 45, 'long_mean_entropy_nat': 1.6700378401420524, 'short_mean_entropy_nat': 1.5266764008162184, 'entropy_relative_change': -0.08584322814723755, 'exp_delta_entropy_change': -0.13355915350198055, 'long_mean_bucket_sigma_c': 1.5819342954324451, 'short_mean_bucket_sigma_c': 1.4088396004415262, 'bucket_sigma_relative_change': -0.10941964877473054, 'truth_pairs': 155, 'median_center_error_long_c': 0.7277007230000017, 'median_center_error_short_c': 0.6401522719999946}, {'cohort': '2026-08-28', 'pairs': 43, 'stations': 43, 'long_mean_entropy_nat': 1.9096708582008755, 'short_mean_entropy_nat': 1.8272093722384128, 'entropy_relative_change': -0.043180994048445887, 'exp_delta_entropy_change': -0.07915309729397013, 'long_mean_bucket_sigma_c': 1.9844291441811555, 'short_mean_bucket_sigma_c': 1.9043489204210113, 'bucket_sigma_relative_change': -0.04035428727448365, 'truth_pairs': 43, 'median_center_error_long_c': 0.7870495843999983, 'median_center_error_sh

## audit.py
完整可审计源码；下一格仅显示，不启动查询。

In [5]:
print((p/'audit.py').read_text())


"""Bounded, read-only diagnosis. No artifacts or activations are published."""
from __future__ import annotations
import collections, datetime as dt, gc, hashlib, json, math, os, pathlib, resource
import shutil, sqlite3, sys, threading, time
import psutil

ROOT = pathlib.Path(__file__).resolve().parents[2]
OUT = pathlib.Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT))
from bayes_basket_target import native_ensemble_oos as oos

ASOF = '2026-09-09T16:36:32Z'
SYSTEMS = {'ECMWF_IFS_ENS_OPEN_PF50', 'ECMWF_AIFS_ENS_GLOBAL',
           'DWD_ICON_GLOBAL_EPS', 'NOAA_GEFS_05', 'ECCC_GEM_GLOBAL_ENS'}
START = time.monotonic()
process = psutil.Process()
def guard():
    while True:
        if (process.memory_info().rss > 850*1024**2 or
            shutil.disk_usage(ROOT).free < 5*1024**3 or
            time.monotonic()-START > 900):
            os.write(2, b'FAIL_CLOSED_RESOURCE_LIMIT\n'); os._exit(73)
        time.sleep(.25)
threading.Thread(target=guard, daemon=True).start()
resource

## snapshot.py
完整可审计源码；下一格仅显示，不启动查询。

In [6]:
print((p/'snapshot.py').read_text())


"""Read four closed event dates, latest served P0 per station/date/lead."""
import sys,pathlib,sqlite3,json,time,math,collections,threading,os,shutil,resource
import psutil
ROOT=pathlib.Path(__file__).resolve().parents[2]; OUT=pathlib.Path(__file__).resolve().parent
sys.path.insert(0,str(ROOT))
from bayes_basket_target.center_width_calibration import bucket_centers_from_intervals
START=time.monotonic();proc=psutil.Process()
def guard():
    while True:
        if proc.memory_info().rss>400*1024**2 or shutil.disk_usage(ROOT).free<5*1024**3 or time.monotonic()-START>600: os._exit(73)
        time.sleep(.25)
threading.Thread(target=guard,daemon=True).start()
resource.setrlimit(resource.RLIMIT_FSIZE,(40*1024**2,40*1024**2))
c=sqlite3.connect(f'file:{ROOT}/data/bayes_basket_target_ssot.db?mode=ro',uri=True,timeout=2)
c.execute('pragma query_only=on');c.execute('pragma cache_size=-4096');c.execute('pragma temp_store=memory')
deadline=time.monotonic()+20;c.set_progress_handler(lambda:int(time

## analyze.py
完整可审计源码；下一格仅显示，不启动查询。

In [7]:
print((p/'analyze.py').read_text())


"""Retrospective chronological sensitivity, NOT strict-as-of promotion OOS.

Retained historical revisions are read at the audit cutoff, not recovered at
each old decision. Gaussian scores isolate center and variance; production
member mixtures and downstream shaping are not replaced by this screen.
"""
import collections,json,math,pathlib,sys,time,os,threading,shutil
import numpy as np
from scipy.special import ndtr
import psutil
ROOT=pathlib.Path(__file__).resolve().parents[2];OUT=pathlib.Path(__file__).resolve().parent
sys.path.insert(0,str(ROOT))
from bayes_basket_target import native_ensemble_oos as oos
proc=psutil.Process();START=time.monotonic()
def guard():
    while True:
        if proc.memory_info().rss>700*1024**2 or time.monotonic()-START>300 or shutil.disk_usage(ROOT).free<5*1024**3:os._exit(73)
        time.sleep(.25)
threading.Thread(target=guard,daemon=True).start()
rows=[json.loads(s) for s in (OUT/'cases.jsonl').read_text().splitlines()]
scopes=collections.defaultdic

## serving_analysis.py
完整可审计源码；下一格仅显示，不启动查询。

In [8]:
print((p/'serving_analysis.py').read_text())


import pathlib,sys,json,collections,math,sqlite3
import numpy as np
ROOT=pathlib.Path(__file__).resolve().parents[2];OUT=pathlib.Path(__file__).resolve().parent
sys.path.insert(0,str(ROOT))
from bayes_basket_target.native_ensemble_p0 import _bucket_centers
from bayes_basket_target.center_width_calibration import apply_center_width_calibration,bucket_centers_from_intervals
from bayes_basket_target.center_width_store import _pre_band_claim,_cell_nll
from bayes_basket_target.cross_consensus import apply_band_reliability
x=json.loads((OUT/'serving_snapshot.json').read_text());rows=x['rows'];truth={}
for line in (OUT/'cases.jsonl').open():
    r=json.loads(line);key=(r['station'],r['event_date'])
    if key in truth:assert truth[key]==r['actual_tmax_c']
    truth[key]=r['actual_tmax_c']
for r in rows:
    intervals=r['intervals'];keys=sorted(intervals,key=lambda k:float('-inf') if intervals[k][0] is None else intervals[k][0])
    cs=_bucket_centers(intervals,keys);p=r['probabilities'];mu=su

## exact_bucket_screen.py
完整可审计源码；下一格仅显示，不启动查询。

In [9]:
print((p/'exact_bucket_screen.py').read_text())


"""Exact single-system member-kernel bucket screen on retained market axes.
Four event dates; preserves native/republisher boundary. Does not claim an
end-to-end fused/served replay or strict historical publication eligibility.
"""
import pathlib,json,sys,math,collections,time,os,threading,shutil
import numpy as np
from scipy.special import ndtr
import psutil
ROOT=pathlib.Path(__file__).resolve().parents[2];OUT=pathlib.Path(__file__).resolve().parent
sys.path.insert(0,str(ROOT))
from bayes_basket_target.native_ensemble_p0 import _interval_probability
proc=psutil.Process();START=time.monotonic()
def guard():
    while True:
        if proc.memory_info().rss>600*1024**2 or time.monotonic()-START>180 or shutil.disk_usage(ROOT).free<5*1024**3:os._exit(73)
        time.sleep(.25)
threading.Thread(target=guard,daemon=True).start()
source=[json.loads(s) for s in (OUT/'cases.jsonl').open()]
case={(r['station'],r['system'],r['event_date'],r['lead'],r['cycle']):r for r in source}
pred=[json.load